In [51]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()                                   # ① read the .env file in this folder
connection_string = os.getenv("CON_STRING")     # ② take the value labelled CON_STRING
weather_key = os.getenv("WEATHER_KEY")          # ③ take the value labelled WEATHER_KEY

# check WITHOUT printing the secrets themselves
print("connection string loaded:", connection_string is not None)
print("weather key loaded:", weather_key is not None)

connection string loaded: True
weather key loaded: True


In [52]:
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,city_name,country,latitude,longitude
0,1,Berlin,Germany,52.52000,13.40500
1,2,Hamburg,Germany,53.55000,10.00000
2,3,Munich,Germany,48.13750,11.57500
3,4,Frankfurt,Germany,50.11056,8.68222


In [9]:
url = "https://api.openweathermap.org/data/2.5/forecast"
params = {"lat": 52.52, "lon": 13.405, "appid": weather_key, "units": "metric"}

response = requests.get(url, params=params)
response.status_code

200

In [10]:
with open(".gitignore", "a") as f:
    f.write(".env\n")

print(open(".gitignore").read())

.env



In [14]:
weather_json = response.json()
weather_json.keys()

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

In [12]:
weather_json["cnt"]

40

In [15]:
type(weather_json["list"])

list

In [16]:
len(weather_json["list"])

40

In [45]:
weather_json["list"][30]#.keys()

{'dt': 1792756800,
 'main': {'temp': 13.21,
  'feels_like': 12.41,
  'temp_min': 13.21,
  'temp_max': 13.21,
  'pressure': 1009,
  'sea_level': 1009,
  'grnd_level': 1003,
  'humidity': 70,
  'temp_kf': 0,
  'dew_point': 7.77},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04d'}],
 'clouds': {'all': 100},
 'wind': {'speed': 3.76, 'deg': 139, 'gust': 8.51},
 'visibility': 10000,
 'pop': 0,
 'sys': {'pod': 'd'},
 'dt_txt': '2026-10-23 12:00:00'}

In [40]:
weather_json["list"][30]['dt_txt']#.keys()

'2026-10-19 18:00:00'

In [42]:
weather_json["list"][30]['wind']['speed']

1.79

In [34]:
weather_json["list"][30]['main']['temp']

7.33

In [47]:
weather_json["list"][30]['pop']

0

In [37]:
weather_json["list"][30]#['main']#['feels_like']

{'dt': 1792432800,
 'main': {'temp': 7.33,
  'feels_like': 6.29,
  'temp_min': 7.33,
  'temp_max': 8.02,
  'pressure': 1030,
  'sea_level': 1030,
  'grnd_level': 1025,
  'humidity': 78,
  'temp_kf': -0.69,
  'dew_point': 3.75},
 'weather': [{'id': 802,
   'main': 'Clouds',
   'description': 'scattered clouds',
   'icon': '03n'}],
 'clouds': {'all': 48},
 'wind': {'speed': 1.79, 'deg': 66, 'gust': 3.41},
 'visibility': 10000,
 'pop': 0,
 'sys': {'pod': 'n'},
 'dt_txt': '2026-10-19 18:00:00'}

In [36]:
weather_json["list"][30]['main']['feels_like']

6.29

In [30]:
weather_json["list"][30]['weather'][0]['description']

'scattered clouds'

In [48]:
rows = []
for forecast in weather_json["list"]:        # forecast = one of the 40 time slots
    rows.append({
        "forecast_time": forecast["dt_txt"],
        "outlook":       forecast["weather"][0]["description"],
        "temperature":   forecast["main"]["temp"],
        "feels_like":    forecast["main"]["feels_like"],
        "wind_speed":    forecast["wind"]["speed"],
        "rain_prob":     forecast["pop"]
    })

weather_df = pd.DataFrame(rows)
weather_df

,forecast_time,outlook,temperature,feels_like,wind_speed,rain_prob
0,2026-10-19 18:00:00,scattered clouds,7.33,6.29,1.79,0.00
1,2026-10-19 21:00:00,broken clouds,7.65,6.37,2.09,0.00
2,2026-10-20 00:00:00,few clouds,7.44,5.59,2.77,0.00
3,2026-10-20 03:00:00,broken clouds,6.15,4.00,2.84,0.00
4,2026-10-20 06:00:00,overcast clouds,6.81,4.78,2.85,0.00
5,2026-10-20 09:00:00,overcast clouds,8.79,6.96,3.16,0.00
6,2026-10-20 12:00:00,overcast clouds,10.72,9.78,3.96,0.00
7,2026-10-20 15:00:00,overcast clouds,10.47,9.56,2.10,0.00
8,2026-10-20 18:00:00,overcast clouds,10.01,9.13,2.78,0.00
9,2026-10-20 21:00:00,overcast clouds,9.98,8.79,2.51,0.00


In [49]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   forecast_time  40 non-null     str    
 1   outlook        40 non-null     str    
 2   temperature    40 non-null     float64
 3   feels_like     40 non-null     float64
 4   wind_speed     40 non-null     float64
 5   rain_prob      40 non-null     float64
dtypes: float64(4), str(2)
memory usage: 2.0 KB


In [50]:
weather_df["forecast_time"] = pd.to_datetime(weather_df["forecast_time"])
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   forecast_time  40 non-null     datetime64[us]
 1   outlook        40 non-null     str           
 2   temperature    40 non-null     float64       
 3   feels_like     40 non-null     float64       
 4   wind_speed     40 non-null     float64       
 5   rain_prob      40 non-null     float64       
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 2.0 KB


In [56]:
def get_weather(cities_df):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    rows = []

    for _, city in cities_df.iterrows():                       # one city at a time
        params = {"lat":   city["latitude"],
                  "lon":   city["longitude"],
                  "appid": weather_key,
                  "units": "metric"}
        response = requests.get(url, params=params)

        if response.status_code != 200:                        # safety: skip a failed city
            print(f"⚠️ Skipped {city['city_name']}: status {response.status_code}")
            continue

        for forecast in response.json()["list"]:               # the 40 time slots
           rows.append({
                "city_id":       city["city_id"],
                "forecast_time": forecast["dt_txt"],
                "outlook":       forecast["weather"][0]["description"],
                "temperature_c": forecast["main"]["temp"],
                "feels_like_c":  forecast["main"]["feels_like"],
                "wind_speed_ms": forecast["wind"]["speed"],
                "rain_prob":     forecast["pop"],
                "rain_3h_mm":    forecast.get("rain", {}).get("3h", 0),
                "snow_3h_mm":    forecast.get("snow", {}).get("3h", 0)
            })

    weather_df = pd.DataFrame(rows)
    weather_df["forecast_time"] = pd.to_datetime(weather_df["forecast_time"])
    weather_df["retrieved_at"] = pd.Timestamp.now().floor("s")   # when we collected it (to the second)
    return weather_df

In [57]:
weather_df = get_weather(cities_df)
weather_df.columns

Index(['city_id', 'forecast_time', 'outlook', 'temperature_c', 'feels_like_c',
       'wind_speed_ms', 'rain_prob', 'rain_3h_mm', 'snow_3h_mm',
       'retrieved_at'],
      dtype='str')

In [55]:
weather_df.columns

Index(['city_id', 'forecast_time', 'outlook', 'temperature', 'feels_like',
       'wind_speed', 'rain_prob', 'retrieved_at'],
      dtype='str')

In [58]:
def load_weather(weather_df):
    """Append the forecasts to the weather table."""
    weather_df.to_sql("weather",
                      if_exists="append",
                      con=connection_string,
                      index=False)
    print(f"✅ {len(weather_df)} weather rows added")


def run_weather_pipeline():
    """Read cities from MySQL, get their forecasts, store them."""
    cities_df = pd.read_sql("cities", con=connection_string)   # fresh read every run → new cities included
    print(f"🏙️ {len(cities_df)} cities found in the database")
    weather_df = get_weather(cities_df)
    load_weather(weather_df)

In [59]:
run_weather_pipeline()

🏙️ 4 cities found in the database
✅ 160 weather rows added
